In [5]:
import pandas as pd
import os
import io
import re
from io import BytesIO
from ftplib import FTP

In [ ]:
def ConnectFTP(server, username, password):
    """
    Descripcion: Esta funcion establece una conexion FTP con el servidor especificado.
    Parametros:
    server (str): La direccion del servidor FTP.
    username (str): El nombre de usuario para la conexion FTP.
    password (str): La contrasena para la conexion FTP.
    Retorna: Un objeto FTP conectado al servidor.
    """
    ftp = FTP(timeout=60)                
    ftp.connect(server, 21)               
    ftp.login(user=username, passwd=password)

    ftp.set_pasv(True)                    
    ftp.voidcmd("TYPE I")                 

    print(f"Conectado a {server}")
    return ftp

def UploadCsvToFTP(df, path):
    """
    Descripcion: Esta funcion sube un DataFrame de pandas como un archivo CSV a un servidor FTP.
    Parametros:
    df (DataFrame): El DataFrame que se desea subir.
    path (str): La ruta en el servidor FTP donde se guardará el archivo CSV.
    Retorna: None
    """
    csv_buffer = io.BytesIO()
    df.to_csv(csv_buffer, index=False, encoding='utf-8')
    csv_buffer.seek(0)
    ftp = ConnectFTP(os.getenv('ftp_server'),os.getenv('ftp_user'),os.getenv('ftp_password'))
    # remote_path = f"/pruebas/csv/{path}"
    ftp.storbinary(f"STOR {path}", csv_buffer)
    ftp.quit()
    print("Archivo subido correctamente:", path)

def ReadCsvFromFTP(aux, namearc ):
    '''
    Descripcion: Esta funcion lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
    Parametros:
    ftp (FTP): Un objeto FTP conectado al servidor.
    remote_file_path (str): La ruta del archivo CSV en el servidor FTP.
    Retorna: Un DataFrame de pandas que contiene los datos del archivo CSV.
    '''
    ftp = ConnectFTP(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
    with BytesIO() as bio:
        ftp.retrbinary(f'RETR /pruebas/csv/{aux}/{namearc}', bio.write)
        bio.seek(0)
        df = pd.read_csv(bio)
    return df

In [7]:
def DrogaImpacto ():
    '''
    Description: Procesa los datos de drogas de impacto, combinando información histórica y de entrevistas iniciales.
    Parameters: None
    Returns: None
    '''
    # Grupos:
    # Droga de impacto sustancia ilegal 
    dfhistorico = ReadCsvFromFTP('HistoricoEISets', 'ECERIECS(SI)_DrogaImpactoGrupos.csv')
    dfhistorico = dfhistorico[dfhistorico['Año'] <= 2020]  # Filtrar por año mayor o igual a 2020
    dfentrevista = ReadCsvFromFTP('EntrevistaInicialSets', 'GPOS-DI-SI.csv')
    df_empalmado = pd.concat([dfhistorico, dfentrevista], axis=0, ignore_index=True)
    UploadCsvToFTP(df_empalmado, '/pruebas/csv/EntrevistaInicialSets/GPOS-DI-SI.csv')
    # Droga de impacto sustancias legales ilegales.
    dfhistorico = ReadCsvFromFTP('HistoricoEISets', 'ECERIECS_DrogaImpactoGrupos.csv')
    dfhistorico = dfhistorico[dfhistorico['Año'] <= 2020]  # Filtrar por año mayor o igual a 2020
    dfentrevista = ReadCsvFromFTP('EntrevistaInicialSets', 'GPOS-DI-SLI.csv')
    df_empalmado = pd.concat([dfhistorico, dfentrevista], axis=0, ignore_index=True)
    UploadCsvToFTP(df_empalmado, '/pruebas/csv/EntrevistaInicialSets/GPOS-DI-SLI.csv')

def AlgunaVez ():
    '''
    Description: Procesa los datos de uso de drogas "Alguna vez", combinando información histórica y de entrevistas iniciales.
    Parameters: None
    Returns: None
    '''
    # Grupos:
    # Alguna vez sustancia ilegal
    dfhistorico = ReadCsvFromFTP('HistoricoEISets', 'ECERIECS(SI)_AlgunaVezGrupos.csv')
    dfhistorico = dfhistorico[dfhistorico['Año'] <= 2020]  # Filtrar por año mayor o igual a 2020
    dfentrevista = ReadCsvFromFTP('EntrevistaInicialSets', 'GPOS-AV-SI.csv')
    df_empalmado = pd.concat([dfhistorico, dfentrevista], axis=0, ignore_index=True)
    UploadCsvToFTP(df_empalmado, '/pruebas/csv/EntrevistaInicialSets/GPOS-AV-SI.csv')
    # Alguna vez sustancias legales ilegales.
    dfhistorico = ReadCsvFromFTP('HistoricoEISets', 'ECERIECS_AlgunaVezGrupos.csv')
    dfhistorico = dfhistorico[dfhistorico['Año'] <= 2020]  # Filtrar por año mayor o igual a 2020
    dfentrevista = ReadCsvFromFTP('EntrevistaInicialSets', 'GPOS-AV-SLI.csv')
    df_empalmado = pd.concat([dfhistorico, dfentrevista], axis=0, ignore_index=True)
    UploadCsvToFTP(df_empalmado, '/pruebas/csv/EntrevistaInicialSets/GPOS-AV-SLI.csv')
def UltimoMes ():
    '''
    Description: Procesa los datos de uso de drogas en el último mes, combinando información histórica y de entrevistas iniciales.
    Parameters: None
    Returns: None
    '''
    # Grupos:
    # Ultimo mes sustancia ilegal
    dfhistorico = ReadCsvFromFTP('HistoricoEISets', 'ECERIECS(SI)_UltimoMesGrupos.csv')
    dfhistorico = dfhistorico[dfhistorico['Año'] <= 2020]  # Filtrar por año mayor o igual a 2020
    dfentrevista = ReadCsvFromFTP('EntrevistaInicialSets', 'GPOS-UM-SI.csv')
    df_empalmado = pd.concat([dfhistorico, dfentrevista], axis=0, ignore_index=True)
    UploadCsvToFTP(df_empalmado, '/pruebas/csv/EntrevistaInicialSets/GPOS-UM-SI.csv')
    # Ultimo mes sustancias legales ilegales.
    dfhistorico = ReadCsvFromFTP('HistoricoEISets', 'ECERIECS_UltimoMesGrupos.csv')
    dfhistorico = dfhistorico[dfhistorico['Año'] <= 2020]  # Filtrar por año mayor o igual a 2020
    dfentrevista = ReadCsvFromFTP('EntrevistaInicialSets', 'GPOS-UM-SLI.csv')
    df_empalmado = pd.concat([dfhistorico, dfentrevista], axis=0, ignore_index=True)
    UploadCsvToFTP(df_empalmado, '/pruebas/csv/EntrevistaInicialSets/GPOS-UM-SLI.csv')

def MotivoConsulta ():
    listelim = ['ConsumoDeDrogas', 'ConsumoDeBebidasAlcoholicas', 'ConsumoDeTabaco','Ludopatia', 'Otro', 'Depresion', 'Psicosis', 'Epilepsia','TrastornosMentales', 'Demencia', 'Autolesion', 'Suicidio', 'Ansiedad']
    listsocio = ['SexoId', 'PerteneceComunidadLGBTTTI', 'PerteneceComunidadIndigena', 'PoblacionAfromexicanaAfroamericana', 'DiscapacidadPerceptual', 'ComunEstadoCivilId', 'ComunEscolaridadId', 'ComunOcupacionId', 'ComunEstratoSocialId', 'Migracion', 'Edad', 'Año', 'Mes', 'Semestre', 'Estado', 'CentroCostoId']
    dfhistorico = ReadCsvFromFTP('HistoricoEISets', 'ECERIECS_DrogaImpactoGrupos.csv')
    dfentrevista = ReadCsvFromFTP('EntrevistaInicialSets', 'GPOS-DI-SLI.csv')
    df_empalmado = pd.concat([dfhistorico, dfentrevista], axis=0, ignore_index=True)
    dfmotivo = df_empalmado [['FolioId'] + listsocio + listelim ]
    UploadCsvToFTP(dfmotivo, '/pruebas/csv/carter/Motivos_Consulta.csv')

In [8]:
def main():
    try:
        DrogaImpacto()
    except Exception as e:
        print("Error en DrogaImpacto", e)    
        return
    print("Droga de impacto procesado exitosamente.")
    try:            
        AlgunaVez()
    except Exception as e:
        print("Error en AlgunaVez", e)
        return
    print("Alguna vez procesado exitosamente.")
    try:
        UltimoMes()
    except Exception as e:
        print("Error en UltimoMes", e)
        return
    print("Ultimo mes procesado exitosamente.")
    try:
        MotivoConsulta()
    except Exception as e:
        print("Error en MotivoConsulta", e)
        return
    print("Motivo de consulta procesado exitosamente.")

    print("Todos los procesos se completaron exitosamente.")
if __name__ == "__main__":
    main()

Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_20836\2510443178.py:49: DtypeWarning: Columns (0,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Conectado a 192.168.62.13
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-DI-SI.csv
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_20836\2510443178.py:49: DtypeWarning: Columns (0,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Conectado a 192.168.62.13
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-DI-SLI.csv
Droga de impacto procesado exitosamente.
Conectado a 192.168.62.13


...

Conectado a 192.168.62.13
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-UM-SLI.csv
Ultimo mes procesado exitosamente.
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_20836\2510443178.py:49: DtypeWarning: Columns (0,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_20836\2510443178.py:49: DtypeWarning: Columns (0,17,22,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/carter/Motivos_Consulta.csv
Motivo de consulta procesado exitosamente.
Todos los procesos se completaron exitosamente.
